In [2]:
# !pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 

In [93]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import os
import pandas as pd
import re

### Loading a base model

In [56]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

### Adding LoRA adapters

In [57]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

### Loading the dataset

In [74]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("a2m2a2n2/bhagwad-gita-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'bhagwad-gita-dataset' dataset.
Path to dataset files: /kaggle/input/bhagwad-gita-dataset


In [71]:
for f in os.listdir(path):
    print(f)

Bhagwad_Gita.csv


In [85]:
data = pd.read_csv(os.path.join(path, "Bhagwad_Gita.csv"))  # replace with actual filename
data.head()
data = data['EngMeaning']
data.head()

,EngMeaning
0,1.1 Dhritarashtra said What did my people and...
1,1.2. Sanjaya said Having seen the army of the...
2,"1.3. ""Behold, O Teacher! this mighty army of t..."
3,"1.4. Here are heroes, mighty archers, eal in b..."
4,"1.5. ""Dhrishtaketu, chekitana and the valiant ..."


In [92]:
text = "	1.1 Dhritarashtra said What did my people and..."

re.sub(r"^\s*\d+\.\d+\s*", "", text)

'Dhritarashtra said What did my people and...'

In [105]:
data = [re.sub(r"^\s*\d+\.\d+\s*", "", i) for i in data]

In [106]:
data

['Dhritarashtra said  What did my people and the sons of Pandu do when they had assembled\ntogether eager for battle on the holy plain of Kurukshetra, O Sanjaya.',
 '. Sanjaya said  Having seen the army of the Pandavas drawn up in battle-array,\nKing Duryodhana then approached his teacher (Drona) and spoke these words.',
 '. "Behold, O Teacher! this mighty army of the sons of Pandu,\narrayed by the son of Drupada, thy wise disciple.',
 '. Here are heroes, mighty archers, eal in battle to Bhima\nand Arjuna, Yoyudhana (Satyaki), Virata and Drupada, of the great car (mighty\nwarriors).',
 '. "Dhrishtaketu, chekitana and the valiant king of Kasi, Purujit\nand Kuntibhoja and Saibya, the best men.',
 '. "The strong Yodhamanyu and the brave Uttamaujas, the son\nof Subhadra (Abhimanyu, the son of Subhadra and Arjuna), and the sons of\nDraupadi, all of great chariots (great heroes).',
 '. "Know also, O best among the twice-born! the names of those\nwho are the most distinguished amongst ourselv

In [60]:
def format_prompt(example):
    return {"text": f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"}

dataset = dataset.map(format_prompt)

In [61]:
print(dataset[0])

{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.', 'input': '', 'instruction': 'Give three tips for staying healthy.', 'text': '### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of 

In [62]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 51760
})

### Train

In [63]:
trainer = SFTTrainer(model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    formatting_func = format_prompt, # Add this line
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,        # small run for testing
        learning_rate = 2e-4,
        output_dir = "outputs",
        logging_steps = 1,
    ),
)

trainer.train()

Unsloth: not enough free memory for dataset tokenization workers (~1GB each); tokenizing in-process.


Unsloth: Tokenizing ["text"]:   0%|          | 0/51760 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 51,760 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
1,1.488919
2,1.770598
3,1.821837
4,1.710012
5,1.287591
6,1.315940
7,1.376198
8,1.478132
9,1.236818
10,1.402924


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


TrainOutput(global_step=60, training_loss=1.289132046699524, metrics={'train_runtime': 107.7733, 'train_samples_per_second': 4.454, 'train_steps_per_second': 0.557, 'total_flos': 840751107059712.0, 'train_loss': 1.289132046699524, 'epoch': 0.00927357032457496})

### Testing Fine Tuned Model

In [64]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151654)
        (layers): ModuleList(
          (0): Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
       

In [65]:
inputs = tokenizer(["### Instruction:\nExplain gravity simply\n\n### Response:\n"], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0]))

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Explain gravity simply

### Response:
Gravity is the force that attracts two objects with mass. It causes things to fall down and makes apples fall from trees, stars orbit around a planet, and planets move in elliptical orbits around the sun. The strength of gravity depends on how much mass an object has - the more massive something is, the stronger its gravitational pull will be. Gravity also helps keep us grounded by pulling our feet towards the ground when we stand or walk. Without gravity, there would be no weight, and everything would float


### Comparing to Instruct model

In [66]:
instruct_model, instruct_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [67]:
inputs = instruct_tokenizer(["### Instruction:\nExplain gravity simply\n\n### Response:\n"], return_tensors="pt").to("cuda")
outputs = instruct_model.generate(**inputs, max_new_tokens=100)
print(instruct_tokenizer.decode(outputs[0]))

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Explain gravity simply

### Response:
Gravity is a force that attracts two objects with mass. It pulls things towards the center of Earth, keeping us on the ground and holding our feet to the floor. Gravity also causes objects to fall down if they are dropped or thrown upwards, and it keeps planets in orbit around stars. Gravity is an important concept in physics because it explains how everything in the universe interacts with each other. Without gravity, there would be no weight, no height, and no distance. Understanding gravity is essential for many fields
